# 🔴 Solution: Ring Attention

Sequence-parallel attention via ring communication, simulated with online softmax

In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def ring_attention(Q, K, V, num_devices):
    """
    环形注意力（单机模拟）。

    核心思想：将 Q/K/V 分布在环形拓扑的设备上，每个设备持有一个 Q 分块，
    K/V 分块在环上轮转。用 online softmax 算法累加注意力输出，
    无需一次性存储完整的注意力矩阵。

    参数:
        Q, K, V: (B, S, D)，S 可被 num_devices 整除
        num_devices: 虚拟环参与者数量

    返回: (B, S, D)，数值上等价于标准注意力
    """
    B, S, D = Q.shape
    chunk = S // num_devices  # 每个设备的序列长度
    scale = D ** -0.5

    # ---- Step 1: 沿序列维度分块 ----
    # Q_chunks: num_devices 个 (B, chunk, D) 的张量
    # 每个设备 i 持有 Q_chunks[i]
    Q_chunks = Q.split(chunk, dim=1)
    K_chunks = K.split(chunk, dim=1)
    V_chunks = V.split(chunk, dim=1)

    outputs = []

    # ---- Step 2: 对每个 Q 分块，遍历所有 K/V 分块 ----
    for i, Qi in enumerate(Q_chunks):
        # ---- Step 3: 初始化 online softmax 的状态 ----
        # m: 运行最大值 (用于数值稳定的 softmax)
        # l: 运行归一化因子 (sum of exp)
        # o: 运行加权输出
        m = torch.full((B, chunk, 1), float('-inf'), device=Q.device, dtype=Q.dtype)
        l = torch.zeros(B, chunk, 1, device=Q.device, dtype=Q.dtype)
        o = torch.zeros(B, chunk, D, device=Q.device, dtype=Q.dtype)

        # 环形遍历: 从第 i 个 K/V 块开始，绕环一圈
        for j in range(num_devices):
            idx = (i + j) % num_devices  # 环形索引
            Kj = K_chunks[idx]  # (B, chunk, D)
            Vj = V_chunks[idx]  # (B, chunk, D)

            # ---- Step 4: 计算局部注意力分数 ----
            # Qi @ Kj^T: (B, chunk, D) @ (B, D, chunk) = (B, chunk, chunk)
            scores = (Qi @ Kj.transpose(-2, -1)) * scale

            # ---- Step 5: Online Softmax 更新 ----
            # 关键公式（数值稳定的在线 softmax）:
            #   m_new = max(m, max(scores))
            #   l_new = l * exp(m - m_new) + sum(exp(scores - m_new))
            #   o_new = (o * l * exp(m - m_new) + exp(scores - m_new) @ Vj) / l_new
            #
            # 直觉: 每次看到新的 K/V 块，更新全局最大值，
            # 之前累积的 l 和 o 要乘以 exp(m - m_new) 做「校正」

            # 新的最大值: 取之前的最大值和当前 scores 最大值的较大者
            m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)

            # 当前块的 exp(scores - m_new)
            exp_scores = torch.exp(scores - m_new)  # (B, chunk, chunk)

            # 更新归一化因子:
            # 旧的 l 要乘以 exp(m - m_new) 校正（因为最大值变了）
            # 加上当前块的 exp 之和
            l_new = l * torch.exp(m - m_new) + exp_scores.sum(dim=-1, keepdim=True)

            # 更新加权输出:
            # 旧的 o 乘以校正因子，加上当前块的贡献
            o = o * (l * torch.exp(m - m_new)) / l_new + (exp_scores @ Vj) / l_new

            # 更新状态
            m, l = m_new, l_new

        outputs.append(o)

    # ---- Step 6: 拼接输出 ----
    return torch.cat(outputs, dim=1)  # (B, S, D)

In [ ]:
# Verify numerical equivalence with standard attention for 2 and 4 devices
torch.manual_seed(42)
B, S, D = 2, 8, 16
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V

for num_devices in [2, 4]:
    ring_out = ring_attention(Q, K, V, num_devices=num_devices)
    max_diff = (ring_out - ref_out).abs().max().item()
    match = torch.allclose(ring_out, ref_out, atol=1e-5)
    print(f'num_devices={num_devices}  shape={tuple(ring_out.shape)}  max_diff={max_diff:.2e}  match={match}')

In [ ]:
from torch_judge import check
check('ring_attention')

## 📝 核心思路总结

1. **序列并行**：把长序列分块到多个设备上，每块只存自己的 Q，K/V 在环上轮转，每个设备都能「看到」所有 K/V。
2. **Online Softmax 是核心**：不需要一次性计算完整 S×S 注意力矩阵，而是逐块累加，用 `m_new = max(m, scores.max)` 更新最大值并校正之前的累积值。
3. **校正因子 `exp(m - m_new)`**：当最大值更新时，之前累积的 l 和 o 都要乘以这个因子来补偿最大值的变化。
4. **环形通信模式**：K/V 块按环形顺序传递，每个 Q 块恰好遍历所有 K/V 块一次，总通信量与设备数成正比。